# Crawl Tiktok data: Video + Comments

# 1. Set up environment

## 1.1 Install dependencies
- If you run this notebook on local, you only need to install yt-dlp

In [66]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
# Import libraries
import os
import requests
import http.cookiejar
import datetime
from datetime import datetime
import re
import json
import time
import pandas as pd
import xlsxwriter
import yt_dlp

In [68]:


cookie_file = "../Crawdata/www.tiktok.com_cookies.txt"

cj = http.cookiejar.MozillaCookieJar(cookie_file)

try:
  # Must have ignore_discard and ignore_expires to install entire cookie file
  cj.load(ignore_discard=True, ignore_expires=True)
  print("Successfully loaded cookie file")
except FileNotFoundError:
  print("Cookie file not found")
except Exception as e:
  print(f"An error occurred: {e}")

# Create fake header: Due to Tiktok may block request without user-agent
headers = {
  'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

url = "https://www.tiktok.com"
response = requests.get(url, cookies=cj, headers=headers)

# Check result
if response.status_code == 200:
  print("Successfully sent request with cookies")
  print(response.text[:500])  # Print first 500 chars of response
else:
  print("An error occurred, status code:", response.status_code)

Cookie file not found
Successfully sent request with cookies
<!doctype html> <html lang="en">   <head>     <link rel="icon" href="data:;base64,=" />     <script id="slardar-config" type="application/json">       {         "slardarClient": "SlardarWAF",         "sdkUrl": "https://sf16-website-login.neutral.ttwstatic.com/obj/tiktok_web_login_static/slardar/fe/sdk-web/browser.sg.js",         "bid": "slardar_us_waf",         "pid": "js-44",         "pluginPathPrefix": "https://sf16-website-login.neutral.ttwstatic.com/obj/tiktok_web_login_static/slardar/fe/sdk


In [69]:
# Create session with cookies and headers
session = requests.Session()
session.cookies = cj
session.headers.update(headers)
print("✅ Session created with cookies and headers")

✅ Session created with cookies and headers


## 1.3 Set up directories

In [70]:
# Define BASE_DIR if not already set
BASE_DIR = globals().get("BASE_DIR", os.getcwd())
BASE_DIR = r"D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results"

if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)
    print(f"📁 Đã setup thư mục mới tại: {BASE_DIR}")
else:
    print(f"✅ Thư mục đã sẵn sàng tại: {BASE_DIR}")

# (Tùy chọn) In ra đường dẫn tuyệt đối để bạn dễ dàng biết chính xác file đang lưu ở đâu trên máy
print("Absolute Output folder:", os.path.abspath(BASE_DIR))

✅ Thư mục đã sẵn sàng tại: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results
Absolute Output folder: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results


# 2. Helper function 
This cell includes helper functions:
1. fetch_oembed:
> - oEmbed is a format for allowing an embedded representation of a URL on third party sites. It allows you to fetch metadata (title, author, description, etc.) of a video by its URL, which is useful for our crawling process

2. extract_aweme_id_from_url:
> - This function extracts the unique identifier (aweme_id) from a TikTok video URL, which is necessary for fetching specific video details.

In [71]:
def fetch_oembed(video_url, session=None, timeout=12):
  "Get metadata oEmbed of video"
  if session is None:
    session = globals().get("session", requests.Session())
  oembed_url = "https://www.tiktok.com/oembed"
  try:
    r = session.get(oembed_url, params={"url": video_url}, timeout=timeout)
    if r.status_code == 200:
      return r.json()
    return None
  except Exception as e:
    print(f"Error fetching oEmbed for {video_url}: {e}")
    return None

def extract_aweme_id_from_url(video_url):
  "Parse aweme_id from URL like /video/<digits>"
  m = re.search(r"/video/(\d+)", video_url)
  return m.group(1) if m else None


# 3. Crawl data
- In this part, we will crawl data by video URLs. You can change the list of URLs to crawl different videos.


##  3.1 Download video

In [72]:
def download_video(video_url, out_dir, write_info_json=True, max_filesize=None):
  os.makedirs(out_dir, exist_ok=True)
  out_template = os.path.join(out_dir, "%(id)s.%(ext)s")

  ydl_opts = {
    "outtmpl": out_template,
    "format": "best",
    "noplaylist": True,
    "cookiesfile": {
      "User-Agent": "Mozilla/5.0",
      "Referer": video_url
    },
    "download_archive": os.path.join(out_dir, "downloaded.txt"),
  }

  if write_info_json:
    ydl_opts.update({
      "writedescriptions": True,
      "writesubtitles": True,
      "writeinfojson": False,
    })
  if max_filesize:
    ydl_opts["max_filesize"] = max_filesize
  with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(video_url, download=True)

  # Get info json
  info_json_path = None
  if isinstance(info, dict) and "id" in info:
    candidate = os.path.join(out_dir, f"{info['id']}.info.json")
    if os.path.exists(candidate):
      info_json_path = candidate
  
  if info_json_path:
    with open(info_json_path, "r", encoding="utf-8") as f:
      info_json = json.load(f)
  else:
    info_data = info
  return info_data


## 3.2 Crawl comments

In [73]:
def fetch_comments_web(aweme_id: str, session: requests.Session, max_comments=200, sleep=1.0):
    url = "https://www.tiktok.com/api/comment/list/"
    cursor, out = 0, []
    while len(out) < max_comments:
        params = {"aid": 1988, "aweme_id": aweme_id, "cursor": cursor, "count": 20}
        r = session.get(url, params=params, timeout=20)
        if r.status_code != 200:
            print("HTTP", r.status_code, r.text[:200])
            break
        try:
            data = r.json()
        except Exception as e:
            print("JSON parse error:", e, r.text[:200]); break

        out.extend(data.get("comments") or [])
        print(f"Fetched {len(out)} comments so far…")

        if not data.get("has_more"):
            break
        cursor = data.get("cursor", cursor + 20)
        if sleep: time.sleep(sleep)
    return out


def fetch_replies_web(aweme_id: str, comment_id: str, session: requests.Session, max_replies=100, sleep=1.0):
    url = "https://www.tiktok.com/api/comment/list/reply/"
    cursor, out = 0, []
    while len(out) < max_replies:
        params = {"aid": 1988, "aweme_id": aweme_id, "comment_id": comment_id, "cursor": cursor, "count": 20}
        r = session.get(url, params=params, timeout=20)
        if r.status_code != 200:
            print("HTTP", r.status_code, r.text[:200])
            break
        try:
            data = r.json()
        except Exception as e:
            print("JSON parse error:", e, r.text[:200]); break

        out.extend(data.get("comments") or [])
        if not data.get("has_more"):
            break
        cursor = data.get("cursor", cursor + 20)
        if sleep: time.sleep(sleep)
    return out

def fetch_all_comments_with_replies_web(aweme_id: str, session: requests.Session,
                                        max_comments=200, max_replies_per_comment=50, sleep=1.0):
    comments = fetch_comments_web(aweme_id, session, max_comments=max_comments, sleep=sleep)
    results = []
    for c in comments:
        item = {
            "cid": c.get("cid"),
            "text": c.get("text"),
            "author": (c.get("user") or {}).get("nickname"),
            "create_time": c.get("create_time"),
            "like_count": c.get("digg_count"),
            "reply_count": c.get("reply_comment_total"),
            "replies": []
        }
        try:
            if item["cid"]:
                rs = fetch_replies_web(aweme_id, item["cid"], session, max_replies=max_replies_per_comment, sleep=sleep)
                item["replies"] = rs
        except Exception as e:
            print(f"⚠️ Reply error for {item['cid']}: {e}")
        results.append(item)
    return results

## 3.3 Crawl 1 video function

In [74]:
def crawl_one_tiktok(video_url, out_dir, session=None, use_comments=False, max_comments=200, max_replies_per_comment=50, sleep=1.0, download_video_file=True):
    if session is None:
        session = globals().get("session", requests.Session())
    
    os.makedirs(out_dir, exist_ok=True)
    result = {
        "url": video_url,
        "crawled_at": datetime.utcnow().isoformat() + "Z"
    }
    # 1) oEmbed

    if download_video_file:
        try:
            video_info = download_video(video_url, out_dir, write_info_json=True)
            result["video_info"] = video_info
            print("✅ Video downloaded successfully")
        except Exception as e:
            print(f"⚠️ Video download error: {e}")
            result["video_info"] = None

    # 2) oEmbed
    oembed = fetch_oembed(video_url, session=session)
    result["oembed"] = oembed

    # 3) Lấy aweme_id
    aweme_id = None
    if oembed and isinstance(oembed, dict):
        aweme_id = oembed.get("embed_product_id")
    if not aweme_id:
        aweme_id = extract_aweme_id_from_url(video_url)

    if not aweme_id:
        raise ValueError("Không lấy được aweme_id cho video này")

    print("aweme_id:", aweme_id)

    # 4) Comments (tuỳ chọn)
    if use_comments:
        cmts = fetch_all_comments_with_replies_web(
            aweme_id, session, max_comments=max_comments,
            max_replies_per_comment=max_replies_per_comment, sleep=sleep
        )
        result["comments"] = cmts

    # 5) Save metadata JSON
    json_path = os.path.join(out_dir, f"{aweme_id}_crawl.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return result, json_path

# 4. Export excel file

In [75]:
# 4. Export to CSV files

def export_videos_metadata_to_csv(all_results, out_dir: str):
    """Export video metadata to CSV"""
    import os, pandas as pd
    os.makedirs(out_dir, exist_ok=True)
    
    rows = []
    for res in all_results:
        video_url = res.get("url", "")
        aweme_id = extract_aweme_id_from_url(video_url) or (res.get("oembed") or {}).get("embed_product_id")
        oembed = res.get("oembed") or {}
        
        rows.append({
            "video_id": aweme_id,
            "video_url": video_url,
            "author_name": oembed.get("author_name", ""),
            "title": oembed.get("title", ""),
            "description": oembed.get("description", ""),
            "upload_date": oembed.get("upload_date", ""),
            "view_count": oembed.get("view_count", ""),
            "like_count": oembed.get("like_count", ""),
            "comment_count": oembed.get("comment_count", ""),
            "crawled_at": res.get("crawled_at", "")
        })
    
    df = pd.DataFrame(rows)
    csv_path = os.path.join(out_dir, "videos_metadata.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"✅ Saved videos metadata CSV: {csv_path}")
    return csv_path, df


def export_comments_to_csv(all_items, aweme_id: str, out_dir: str):
    """Export comments and replies to CSV"""
    import os, pandas as pd
    os.makedirs(out_dir, exist_ok=True)
    
    rows = []
    for c in all_items or []:
        # Top-level comments
        rows.append({
            "video_id": aweme_id,
            "comment_id": c.get("cid"),
            "parent_comment_id": None,
            "is_reply": 0,
            "author": c.get("author"),
            "comment_text": c.get("text"),
            "likes": c.get("like_count"),
            "reply_count": c.get("reply_count"),
            "created_at_timestamp": c.get("create_time"),
        })
        
        # Replies to comment
        for r in (c.get("replies") or []):
            ru = r.get("user") or {}
            rows.append({
                "video_id": aweme_id,
                "comment_id": r.get("cid"),
                "parent_comment_id": c.get("cid"),
                "is_reply": 1,
                "author": ru.get("nickname"),
                "comment_text": r.get("text"),
                "likes": r.get("digg_count") or r.get("like_count"),
                "reply_count": r.get("reply_comment_total") or r.get("reply_count"),
                "created_at_timestamp": r.get("create_time"),
            })
    
    df = pd.DataFrame(rows)
    
    # Convert timestamp to datetime
    if "created_at_timestamp" in df.columns and len(df) > 0:
        df["created_at_utc"] = pd.to_datetime(df["created_at_timestamp"], unit="s", errors="coerce", utc=True)
        df["created_at_vn"] = df["created_at_utc"].dt.tz_convert("Asia/Ho_Chi_Minh")
    
    # Reorder columns
    cols = ["video_id", "comment_id", "parent_comment_id", "is_reply", "author",
            "comment_text", "likes", "reply_count", "created_at_utc", "created_at_vn"]
    df = df[[c for c in cols if c in df.columns]]
    
    csv_path = os.path.join(out_dir, f"{aweme_id}_comments.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"✅ Saved comments CSV: {csv_path}")
    return csv_path, df

# 5. Test on 1 video

In [76]:
# VIDEO_URL = "https://www.tiktok.com/@hesti.home/video/7628593511155109140"  

# res, jsonp = crawl_one_tiktok(
#     VIDEO_URL, BASE_DIR,
#     use_comments=True,         # bật/tắt crawl bình luận
#     max_comments=100,          # tối đa số bình luận cấp 1
#     max_replies_per_comment=50 # tối đa số reply/1 bình luận
# )
# print("Metadata saved to:", jsonp)
# print("Keys:", list(res.keys()))

# # Nếu có comments thì xuất Excel
# cmts = res.get("comments", [])
# if cmts:
#     aweme_id = extract_aweme_id_from_url(VIDEO_URL) or (res.get("oembed") or {}).get("embed_product_id")
#     xlsx_path, df_preview = export_comments_to_excel(cmts, aweme_id, BASE_DIR)
#     print(df_preview.head(3))
# else:
#     print("No comments fetched.")


# 6. Crawl multiple videos

In [ ]:
json_file_path = "../Crawdata/dataset_tiktok-scraper_2026-04-17_10-26-02-949.json"

try:
  with open(json_file_path, "r", encoding="utf-8") as f:
    videos_data = json.load(f)
  print(f"✅ Successfully loaded JSON data from {json_file_path}, total videos: {len(videos_data)}")
except FileNotFoundError:
  print(f"❌ JSON file not found at: {json_file_path}")
  videos_data = []

# Lists to store all data for combined CSV export
all_video_results = []
all_comments_data = []

results_summary = []
for idx, video_info in enumerate(videos_data, 1):
  try:
    video_url = video_info.get("webVideoUrl")
    if not video_url:
      print(f"⚠️ Skipping video #{idx} due to missing URL")
      continue
    
    print(f"\n{'='*60}")
    print(f"🎬 Video {idx}/{len(videos_data)}")
    print(f"{'='*60}")
    print(f"URL: {video_url}")

    video_dir = os.path.join(BASE_DIR, f"video_{idx}")

    # Crawl video
    res, jsonp = crawl_one_tiktok(
      video_url,
      video_dir,
      use_comments=True,
      max_comments=50,
      max_replies_per_comment=20,
      download_video_file=True
    )
    
    # Store video result
    all_video_results.append(res)
    
    # Export comments to CSV
    cmts = res.get("comments", [])
    if cmts:
        aweme_id = extract_aweme_id_from_url(video_url) or (res.get("oembed") or {}).get("embed_product_id")
        csv_path, df = export_comments_to_csv(cmts, aweme_id, video_dir)
        print(f"📊 Extracted {len(df)} comments/replies from this video")
        all_comments_data.append((aweme_id, df))
    else:
        print("⚠️ No comments fetched for this video")

    results_summary.append({
      "index": idx,
      "url": video_url,
      "author": video_info.get("authorMeta.name"),
      "status": "✅ Success",
    })
    print(f"\n✅ Video {idx} crawl successful!")
    
  except Exception as e:
    print(f"❌ Error crawling video {idx}: {e}")
    results_summary.append({
      "index": idx,
      "url": video_info.get("webVideoUrl", "Unknown"),
      "author": video_info.get("authorMeta.name", "Unknown"),
      "status": f"❌ Error: {str(e)}",
    })

# Export combined CSV files
print(f"\n{'='*60}")
print("📁 EXPORTING COMBINED CSV FILES...")
print(f"{'='*60}")

# Export all videos metadata to one CSV
if all_video_results:
    videos_csv_path, df_videos = export_videos_metadata_to_csv(all_video_results, BASE_DIR)
    print(f"📄 Saved {len(df_videos)} videos metadata")

# Combine all comments from all videos
if all_comments_data:
    combined_df = pd.concat([df for _, df in all_comments_data], ignore_index=True)
    combined_csv_path = os.path.join(BASE_DIR, "all_comments.csv")
    combined_df.to_csv(combined_csv_path, index=False, encoding="utf-8")
    print(f"💬 Saved {len(combined_df)} total comments/replies to: {combined_csv_path}")

print(f"\n{'='*60}")
print("📊 SUMMARY RESULTS:")
print(f"{'='*60}")
for item in results_summary:
  print(f"Video #{item['index']:02d}: {item['author']} - {item['status']}")

successful = len([r for r in results_summary if "✅" in r['status']])
print(f"\n✅ Total: {successful}/{len(results_summary)} videos successful")
print(f"📁 All data saved to: {BASE_DIR}")

✅ Successfully loaded JSON data from ../Crawdata/dataset_tiktok-scraper_2026-04-17_10-26-02-949.json, total videos: 26

🎬 Video 1/26
URL: https://www.tiktok.com/@cahebietnhayy/video/7629295002790956308
[TikTok] Extracting URL: https://www.tiktok.com/@cahebietnhayy/video/7629295002790956308
[TikTok] 7629295002790956308: Downloading webpage


C:\Users\KietKlat\AppData\Local\Temp\ipykernel_27184\2560448234.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawled_at": datetime.utcnow().isoformat() + "Z"


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7629295002790956308: Downloading webpage with challenge cookie
[info] 7629295002790956308: Downloading 1 format(s): bytevc1_1080p_897128-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_1\7629295002790956308.mp4
[download] 100% of    2.15MiB in 00:00:01 at 1.16MiB/s   
✅ Video downloaded successfully
aweme_id: 7629295002790956308
Fetched 20 comments so far…
Fetched 40 comments so far…
Fetched 60 comments so far…
✅ Saved comments CSV: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_1\7629295002790956308_comments.csv
📊 Extracted 69 comments/replies from this video

✅ Video 1 crawl successful!

🎬 Video 2/26
URL: https://www.tiktok.com/@nhi.lng167/video/7629213074599316744
[TikTok] Extracting URL: https://www.tiktok.com/@nhi.lng167/video/7629213074599316744
[TikTok] 7629213074599316744: Downl

C:\Users\KietKlat\AppData\Local\Temp\ipykernel_27184\2560448234.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawled_at": datetime.utcnow().isoformat() + "Z"


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7629213074599316744: Downloading webpage with challenge cookie
[info] 7629213074599316744: Downloading 1 format(s): bytevc1_1080p_397450-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_2\7629213074599316744.mp4
[download] 100% of    1.62MiB in 00:00:01 at 1.60MiB/s   
✅ Video downloaded successfully
aweme_id: 7629213074599316744
Fetched 3 comments so far…
✅ Saved comments CSV: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_2\7629213074599316744_comments.csv
📊 Extracted 5 comments/replies from this video

✅ Video 2 crawl successful!

🎬 Video 3/26
URL: https://www.tiktok.com/@thlinhdayne/video/7629363271769869589
[TikTok] Extracting URL: https://www.tiktok.com/@thlinhdayne/video/7629363271769869589
[TikTok] 7629363271769869589: Downloading webpage


...

[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7629287332490775829: Downloading webpage with challenge cookie
[info] 7629287332490775829: Downloading 1 format(s): bytevc1_1080p_1394441-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_9\7629287332490775829.mp4
[download] 100% of    2.49MiB in 00:00:01 at 1.58MiB/s   
✅ Video downloaded successfully
aweme_id: 7629287332490775829
Fetched 20 comments so far…
Fetched 40 comments so far…
Fetched 60 comments so far…
✅ Saved comments CSV: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_9\7629287332490775829_comments.csv
📊 Extracted 73 comments/replies from this video

✅ Video 9 crawl successful!

🎬 Video 10/26
URL: https://www.tiktok.com/@dthquangngai.mcv/video/7629266935846472978
[TikTok] Extracting URL: https://www.tiktok.com/@dthquangngai.mcv/video/7629266935846472978
[TikTok] 762926693584

C:\Users\KietKlat\AppData\Local\Temp\ipykernel_27184\2560448234.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawled_at": datetime.utcnow().isoformat() + "Z"


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7629266935846472978: Downloading webpage with challenge cookie
[info] 7629266935846472978: Downloading subtitles: eng-US
[info] 7629266935846472978: Downloading 1 format(s): bytevc1_1080p_225447-1
[info] Writing video subtitles to: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_10\7629266935846472978.eng-US.vtt
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_10\7629266935846472978.eng-US.vtt
[download] 100% of    1.86KiB in 00:00:00 at 8.96KiB/s   
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_10\7629266935846472978.mp4
[download] 100% of    1.45MiB in 00:00:00 at 1.59MiB/s     
✅ Video downloaded successfully
aweme_id: 7629266935846472978
Fetched 20 comments so far…
Fetched 40 comments so far…
Fetched 60 comments so far…
✅ Saved comments CSV: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok

C:\Users\KietKlat\AppData\Local\Temp\ipykernel_27184\2560448234.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "crawled_at": datetime.utcnow().isoformat() + "Z"


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7627470983200312594: Downloading webpage with challenge cookie
[info] 7627470983200312594: Downloading 1 format(s): bytevc1_1080p_1306138-1
[info] There are no subtitles for the requested languages
[download] Destination: D:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\tiktok-results\video_11\7627470983200312594.mp4
[download] 100% of    2.21MiB in 00:00:01 at 1.62MiB/s   
✅ Video downloaded successfully
